In [1]:
# 0

import re
import time
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

XLSX = 'kospi_data.xlsx'

EXCLUDE_FIN = ['A000680', 'A000880', 'A005110', 'A008060', 'A015020', 'A023590',
               'A026890', 'A030190', 'A034310', 'A210980', 'A244920']

SHEET_ITEM = {
    '01_actq': 'actq', '02_rectq': 'rectq', '03_invtq': 'invtq',
    '04_ppentq': 'ppentq', '05_atq': 'atq', '06_lctq': 'lctq',
    '07_dlttq': 'dlttq', '08_dlcq': 'dlcq', '09_ltq': 'ltq',
    '10_seqq': 'seqq', '11_cogsq': 'cogsq', '12_xsgaq': 'xsgaq',
    '13_saleq': 'saleq', '14_apq': 'apq', '15_opinc': 'opinc',
    '16_xintq': 'xintq', '17_ibq': 'ibq', '18_txditcq': 'txditcq',
    '19_pstkq': 'pstkq', '20_cheq': 'cheq', '21_oancfq': 'oancfq',
    '22_dpq': 'dpq', '23_dvq': 'dvq', '24_cstkq': 'cstkq',
    '25_req': 'req', '26_epsq': 'epsq', '27_txtq': 'txtq',
}

PRICE_SHEET = {'28_return': 'ret', '29_mktcap': 'mktcap', '30_price': 'price'}

RULES = [
    ('부도',     r'부도',                                  '부실'),
    ('정리절차', r'정리|회생|화의|파산',                    '부실'),
    ('자본잠식', r'잠식',                                  '부실'),
    ('감사의견', r'감사의견',                               '부실'),
    ('영업정지', r'영업활동|영업용자산|경매|계속성|폐업',     '부실'),
    ('미제출',   r'미제출',                                '부실'),
    ('주가시총', r'주가수준|시가총액',                      '비부실'),
    ('유동성',   r'거래량|분포',                           '비부실'),
    ('기타부실', r'유예기간',                               '부실'),
    ('합병',     r'합병|완전자회사|주식교환',                '비부실'),
    ('자진',     r'자진|신청에\s*의한|상장폐지\s*신청',       '비부실'),
]
BAD_CAT = {k: b for k, _, b in RULES}


def classify(s):
    for k, p, _ in RULES:
        if re.search(p, s):
            return k
    return '미분류'


def load_long_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    periods = raw.iloc[9, 3:].tolist()
    body = raw.iloc[13:, :].reset_index(drop=True)
    codes = body.iloc[:, 0].astype(str)
    mask = codes.str.startswith('A')
    vals = (body.loc[mask.values, 3:].apply(pd.to_numeric, errors='coerce')
            .reset_index(drop=True))
    vals.columns = periods
    vals.insert(0, 'code', codes[mask].reset_index(drop=True))
    long = vals.melt(id_vars='code', var_name='period', value_name=var)
    long['date'] = (pd.to_datetime(long['period'].astype(int).astype(str), format='%Y%m')
                    + pd.offsets.MonthEnd(0))
    return long[['code', 'date', var]]


def load_wide_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    codes = raw.iloc[7, 1:].astype(str)
    mask = codes.str.startswith('A')
    dates = pd.to_datetime(raw.iloc[14:, 0].tolist())
    vals = raw.iloc[14:, 1:].loc[:, mask.values].apply(pd.to_numeric, errors='coerce')
    vals.columns = codes[mask].values
    vals.index = dates
    long = vals.stack().rename(var).reset_index()
    long.columns = ['date', 'code', var]
    return long[['code', 'date', var]]


def load_single_sheet(path, sheet, var):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    dates = pd.to_datetime(raw.iloc[14:, 0].tolist())
    vals = pd.to_numeric(raw.iloc[14:, 1], errors='coerce')
    return pd.DataFrame({'date': dates, var: vals.values})


def load_all(path=XLSX):
    t0 = time.time()

    fin = None
    for sheet, var in SHEET_ITEM.items():
        long = load_long_sheet(path, sheet, var)
        fin = long if fin is None else fin.merge(long, on=['code', 'date'], how='outer')
    fin = fin[~fin['code'].isin(EXCLUDE_FIN)].sort_values(['code', 'date']).reset_index(drop=True)
    print(f'financials  {fin.shape}  {time.time() - t0:.0f}초')

    t = time.time()
    px = None
    for sheet, var in PRICE_SHEET.items():
        long = load_wide_sheet(path, sheet, var)
        px = long if px is None else px.merge(long, on=['code', 'date'], how='outer')
    px = px[~px['code'].isin(EXCLUDE_FIN)].sort_values(['code', 'date']).reset_index(drop=True)
    print(f'prices      {px.shape}  {time.time() - t:.0f}초')

    t = time.time()
    dr = load_wide_sheet(path, '31_dret', 'dret')
    dr = dr[~dr['code'].isin(EXCLUDE_FIN)].dropna(subset=['dret'])
    dr = dr.sort_values(['code', 'date']).reset_index(drop=True)
    print(f'daily       {dr.shape}  {time.time() - t:.0f}초')

    t = time.time()
    mkt = (load_single_sheet(path, '32_kret', 'kret')
           .merge(load_single_sheet(path, '33_rf', 'rf'), on='date', how='outer')
           .sort_values('date').reset_index(drop=True))

    raw = pd.read_excel(path, sheet_name='34_gd', header=None)
    body = raw.iloc[14:, :2].dropna()
    gd = pd.DataFrame({
        'qtr': pd.PeriodIndex(body.iloc[:, 0].astype(str), freq='Q'),
        'gd': pd.to_numeric(body.iloc[:, 1], errors='coerce'),
    }).dropna().sort_values('qtr').reset_index(drop=True)

    raw = pd.read_excel(path, sheet_name='36_delisted', header=None)
    body = raw.iloc[13:, :].reset_index(drop=True)
    codes = body.iloc[:, 0].astype(str)
    mask = codes.str.startswith('A')
    dl = pd.DataFrame({
        'code': codes[mask].values,
        'name': body.loc[mask.values, 1].astype(str).values,
        'del_date': pd.to_datetime(body.loc[mask.values, 3].astype('Int64').astype(str),
                                   format='%Y%m%d', errors='coerce').values,
        'reason': body.loc[mask.values, 4].astype(str).values,
    })
    dl = dl[~dl['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)
    dl['cat'] = dl['reason'].fillna('').map(classify)
    dl['is_bad'] = dl['cat'].map(BAD_CAT).eq('부실').astype(int)
    dl['excl'] = dl['cat'].eq('미분류').astype(int)
    print(f'market/gd/delisted  {time.time() - t:.0f}초')

    print(f'합계 {time.time() - t0:.0f}초')
    return fin, px, dr, mkt, gd, dl


financials, prices, daily_returns, market, gd, delisted = load_all()

ov = pd.DataFrame({
    '행': [len(financials), len(prices), len(daily_returns), len(market), len(gd), len(delisted)],
    '종목': [financials['code'].nunique(), prices['code'].nunique(),
           daily_returns['code'].nunique(), np.nan, np.nan, delisted['code'].nunique()],
    '시작': [financials['date'].min().date(), prices['date'].min().date(),
           daily_returns['date'].min().date(), market['date'].min().date(),
           str(gd['qtr'].min()), delisted['del_date'].min().date()],
    '끝': [financials['date'].max().date(), prices['date'].max().date(),
          daily_returns['date'].max().date(), market['date'].max().date(),
          str(gd['qtr'].max()), delisted['del_date'].max().date()],
}, index=pd.Index(['financials', 'prices', 'daily_returns', 'market', 'gd', 'delisted'],
                  name='테이블'))
display(ov)

cnt = (delisted.groupby('cat')
       .agg(종목=('code', 'size'), 부실=('is_bad', 'sum'))
       .sort_values('종목', ascending=False))
cnt.index.name = '사유'
display(cnt)


financials  (116235, 29)  20초
prices      (353133, 5)  6초
daily       (4323086, 3)  34초
market/gd/delisted  1초
합계 61초


,행,종목,시작,끝
테이블,,,,
financials,116235,1107.0,2000-03-31,2026-03-31
prices,353133,1107.0,1999-12-28,2026-06-30
daily_returns,4323086,953.0,1999-12-28,2026-06-30
market,6530,NaN,1999-12-28,2026-06-30
gd,105,NaN,2000Q1,2026Q1
delisted,355,355.0,1991-06-13,2026-06-30


,종목,부실
사유,,
합병,99,0
감사의견,69,69
정리절차,33,33
미분류,33,0
부도,30,30
자본잠식,29,29
영업정지,25,25
자진,17,0
주가시총,8,0


In [2]:
# 1

YR0 = 2002
HALT_N = 8
FULL_HALT = 0.95
EXTREME = 100.0

px = prices.copy()
px['date'] = pd.to_datetime(px['date'])
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
px['year'] = px['date'].dt.year

dec_year = px.loc[(px['ym'].dt.month == 12) & px['mktcap'].notna(), 'year']
LAST_YEAR = int(dec_year.max())
TARGET = LAST_YEAR + 1
print(f'설명변수 마지막 연도 {LAST_YEAR}  →  지정 대상 위험군 {TARGET}년')

dc = delisted.copy()
dc['del_date'] = pd.to_datetime(dc['del_date'])
excl_codes = set(dc.loc[dc['excl'] == 1, 'code'])
dcx = dc[dc['excl'] == 0][['code', 'del_date', 'is_bad']].copy()
dcx['del_year'] = dcx['del_date'].dt.year.astype(int)

live = px[px['ret'].notna() | px['mktcap'].notna()][['code', 'year']].drop_duplicates()
alive = pd.concat([live, dcx[['code', 'del_year']].rename(columns={'del_year': 'year'})],
                  ignore_index=True).drop_duplicates()

carry = alive.loc[alive['year'] == TARGET - 1, ['code']].assign(year=TARGET)
alive = pd.concat([alive, carry], ignore_index=True).drop_duplicates()

alive = alive[(alive['year'] >= YR0) & (alive['year'] <= TARGET)]
alive = alive[~alive['code'].isin(excl_codes)]
alive = alive.merge(dcx[['code', 'del_year', 'is_bad']], on='code', how='left')
alive = alive[alive['del_year'].isna() | (alive['year'] <= alive['del_year'])]
alive['y'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 1)).astype(int)
alive['censored'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 0)).astype(int)

riskset = (alive.rename(columns={'year': 'obs_year'})[['code', 'obs_year', 'y', 'censored']]
           .sort_values(['code', 'obs_year']).reset_index(drop=True))

d = daily_returns.copy()
d['date'] = pd.to_datetime(d['date'])
d = d.sort_values(['code', 'date']).reset_index(drop=True)
d['ym'] = pd.PeriodIndex(d['date'], freq='M')
d['year'] = d['date'].dt.year
d['is0'] = (d['dret'] == 0).astype(int)
d['grp'] = (d['is0'] != d.groupby('code')['is0'].shift()).cumsum()
d['runlen_yr'] = d.groupby(['grp', 'year'])['is0'].transform('size')
d['halt'] = ((d['is0'] == 1) & (d['runlen_yr'] >= HALT_N)).astype(int)
d['extreme'] = (d['dret'].abs() > EXTREME).astype(int)
halt_daily = d[['code', 'date', 'ym', 'year', 'dret', 'halt', 'extreme']].copy()

hm = (d.groupby(['code', 'ym'])
      .agg(n_days=('halt', 'size'), n_halt=('halt', 'sum'), n_ext=('extreme', 'sum'))
      .reset_index())
hm['halt_frac'] = hm['n_halt'] / hm['n_days']

pxs = px.sort_values(['code', 'date']).copy()
pxs['p_prev'] = pxs.groupby('code')['price'].shift(1)
pxs['frozen'] = (pxs['price'].notna() & pxs['p_prev'].notna()
                 & ((pxs['price'] - pxs['p_prev']).abs() < 1e-9)).astype(int)
DAILY_CODES = set(d['code'].unique())
pxs['no_daily'] = (~pxs['code'].isin(DAILY_CODES)).astype(int)

hm = pxs[['code', 'ym', 'ret', 'frozen', 'no_daily']].merge(hm, on=['code', 'ym'], how='outer')
for c, v in [('halt_frac', 0.0), ('n_days', 0), ('n_halt', 0), ('n_ext', 0), ('frozen', 0)]:
    hm[c] = hm[c].fillna(v)
hm['no_daily'] = hm['no_daily'].fillna(1).astype(int)
hm['frozen'] = hm['frozen'].astype(int)

hm['by_daily'] = (hm['halt_frac'] >= FULL_HALT).astype(int)
hm['by_price'] = ((hm['no_daily'] == 1) & (hm['frozen'] == 1)).astype(int)
hm['reopen'] = (((hm['by_daily'] == 1) | (hm['by_price'] == 1))
                & hm['ret'].notna() & hm['ret'].abs().gt(1e-9)).astype(int)
hm['full_halt'] = (((hm['by_daily'] == 1) | (hm['by_price'] == 1))
                   & (hm['reopen'] == 0)).astype(int)
hm['src'] = np.select(
    [hm['reopen'] == 1, hm['by_daily'] == 1, hm['by_price'] == 1],
    ['재개월(제외)', '일별', '가격(일별없음)'], default='정지아님')
halt_month = hm[['code', 'ym', 'n_days', 'n_halt', 'n_ext', 'halt_frac', 'frozen',
                 'no_daily', 'by_daily', 'by_price', 'reopen', 'full_halt', 'src']].copy()

r0 = riskset[riskset['obs_year'] <= LAST_YEAR]
rt = riskset[riskset['obs_year'] == TARGET]
ov = pd.DataFrame({'값': [
    len(riskset), riskset['code'].nunique(),
    f"{int(riskset['obs_year'].min())}~{int(riskset['obs_year'].max())}",
    len(r0), int(r0['y'].sum()), int(r0['censored'].sum()),
    len(rt), int(rt['y'].sum())]},
    index=pd.Index(['위험집합 행', '종목', '기간',
                    f'  ~{LAST_YEAR} 행', '  사건', '  절단',
                    f'  {TARGET} 행', f'  {TARGET} 사건(확정분)'], name='항목'))
display(ov)

src = halt_month.groupby('src').agg(월수=('code', 'size'), 종목=('code', 'nunique'))
src.index.name = '판정 출처'
display(src)

hv = pd.DataFrame({'값': [
    HALT_N, FULL_HALT, EXTREME,
    len(halt_daily), int(halt_daily['halt'].sum()),
    round(halt_daily['halt'].mean() * 100, 2),
    int(halt_daily.loc[halt_daily['halt'] == 1, 'code'].nunique()),
    int(halt_daily['extreme'].sum()),
    int(halt_month['full_halt'].sum()),
    int(halt_month.loc[halt_month['by_daily'] == 1, 'full_halt'].sum()),
    int(halt_month.loc[halt_month['by_price'] == 1, 'full_halt'].sum()),
    int(halt_month['reopen'].sum())]},
    index=pd.Index(['정지 기준일수', '완전정지 임계', '극단값 임계(%)',
                    '일별 행', '  정지일', '  정지 비중 %', '  정지 경험 종목', '  극단값',
                    '완전정지 월', '  일별 판정', '  가격 판정', '재개월(제외)'], name='항목'))
display(hv)


설명변수 마지막 연도 2025  →  지정 대상 위험군 2026년


,값
항목,
위험집합 행,17050
종목,936
기간,2002~2026
~2025 행,16289
사건,108
절단,67
2026 행,761
2026 사건(확정분),5


,월수,종목
판정 출처,,
가격(일별없음),268,31
일별,1640,235
재개월(제외),42,39
정지아님,351183,1107


,값
항목,
정지 기준일수,8.00
완전정지 임계,0.95
극단값 임계(%),100.00
일별 행,4323086.00
정지일,48954.00
정지 비중 %,1.13
정지 경험 종목,560.00
극단값,15.00
완전정지 월,1908.00


In [3]:
# 2

VY0, VY1 = 2001, LAST_YEAR
DRET_CAP = 100.0
MIN_M12, MIN_D12 = 3, 20
MIN_MONTHS = 12
ACC_Q = 2

FVOL_COLS = ['actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlcq',
             'apq', 'dlttq', 'ltq', 'seqq', 'xoprq', 'cogsq', 'xsgaq']

mkt = market.copy()
mkt['date'] = pd.to_datetime(mkt['date'])
mkt['ym'] = pd.PeriodIndex(mkt['date'], freq='M')
mkt_m = (mkt.assign(g=1 + mkt['kret'] / 100).groupby('ym')['g'].prod().sub(1)
         .rename('mret').reset_index())
mkt_m['year'] = mkt_m['ym'].dt.year
mkt_d = mkt[['date', 'kret']].dropna().assign(mret_d=lambda x: x['kret'] / 100)[['date', 'mret_d']]
m_ann = (mkt_m.assign(g=1 + mkt_m['mret']).groupby('year')['g'].prod().sub(1)
         .rename('m_ann').reset_index())


def sig_group(df, xcol, minobs):
    uniq, gid = np.unique(df['key'].values, return_inverse=True)
    G = len(uniq)
    y = df['yv'].values.astype('float64')
    n = np.bincount(gid, minlength=G).astype('float64')
    sy = np.bincount(gid, weights=y, minlength=G)
    syy = np.bincount(gid, weights=y * y, minlength=G)
    x = df[xcol].values.astype('float64')
    sx = np.bincount(gid, weights=x, minlength=G)
    sxx = np.bincount(gid, weights=x * x, minlength=G)
    sxy = np.bincount(gid, weights=x * y, minlength=G)
    det = n * sxx - sx * sx
    with np.errstate(divide='ignore', invalid='ignore'):
        b0 = (sxx * sy - sx * sxy) / det
        b1 = (n * sxy - sx * sy) / det
        ssr = syy - b0 * sy - b1 * sxy
        sd = np.sqrt(ssr / (n - 2))
    ok = (n >= max(minobs, 3)) & (det > 0) & (ssr > 0)
    o = pd.DataFrame({'key': uniq, 'n': n.astype(int), 'sig': np.where(ok, sd, np.nan)})
    o['code'] = [k.rsplit('_', 1)[0] for k in o['key']]
    o['year'] = [int(k.rsplit('_', 1)[1]) for k in o['key']]
    return o[['code', 'year', 'n', 'sig']]


dd = halt_daily[(halt_daily['halt'] == 0) & (halt_daily['extreme'] == 0)
                & halt_daily['dret'].notna()].copy()
dd = dd[(dd['dret'].abs() <= DRET_CAP) & dd['year'].between(VY0, VY1)]
dd = dd.merge(mkt_d, on='date', how='inner')
dd['yv'] = dd['dret'] / 100
dd['key'] = dd['code'] + '_' + dd['year'].astype(str)

pm = px.merge(halt_month[['code', 'ym', 'full_halt']], on=['code', 'ym'], how='left')
pm['full_halt'] = pm['full_halt'].fillna(0)
pm = pm[(pm['full_halt'] == 0) & pm['ret'].notna() & pm['year'].between(VY0, VY1)]
pm = pm.merge(mkt_m[['ym', 'mret']], on='ym', how='inner')
pm['yv'] = pm['ret'] / 100
pm['key'] = pm['code'] + '_' + pm['year'].astype(str)

sigma = (sig_group(pm, 'mret', MIN_M12).rename(columns={'sig': 'SIGMA_m12', 'n': 'n_m12'})
         .merge(sig_group(dd, 'mret_d', MIN_D12).rename(columns={'sig': 'SIGMA_d12',
                                                                 'n': 'n_d12'}),
                on=['code', 'year'], how='outer'))
sigma[['n_m12', 'n_d12']] = sigma[['n_m12', 'n_d12']].fillna(0).astype(int)
sigma['SIGMA_m12_a'] = sigma['SIGMA_m12'] * np.sqrt(12)
sigma['SIGMA_d12_a'] = sigma['SIGMA_d12'] * np.sqrt(252)

pxh = px.merge(halt_month[['code', 'ym', 'n_days', 'n_halt', 'full_halt']],
               on=['code', 'ym'], how='left')
for c in ['n_days', 'n_halt', 'full_halt']:
    pxh[c] = pxh[c].fillna(0)
pxh = pxh[pxh['year'].between(VY0, VY1)].copy()
pxh['mret_i'] = pxh['ret'] / 100

hy = (pxh.groupby(['code', 'year'])
      .agg(d_tot=('n_days', 'sum'), d_halt=('n_halt', 'sum'),
           m_tot=('ym', 'size'), m_halt=('full_halt', 'sum')).reset_index())
hy['halt_frac_y'] = np.where(hy['d_tot'] > 0, hy['d_halt'] / hy['d_tot'],
                             hy['m_halt'] / hy['m_tot'])
hy['full_year'] = (hy['m_halt'] / hy['m_tot'] >= FULL_HALT).astype(int)
hy = hy[['code', 'year', 'halt_frac_y', 'full_year']]

rr = pxh[pxh['mret_i'].notna()].copy()
rr['g'] = 1 + rr['mret_i']
fy = rr.groupby(['code', 'year']).agg(g=('g', 'prod'), n_ret=('mret_i', 'count')).reset_index()
fy = fy.merge(m_ann, on='year', how='left')
fy['EXRET'] = np.where(fy['n_ret'] >= MIN_MONTHS, fy['g'] - 1 - fy['m_ann'], np.nan)
fy = fy.merge(hy, on=['code', 'year'], how='left')
fy.loc[fy['full_year'] == 1, 'EXRET'] = np.nan

dec = pxh[(pxh['ym'].dt.month == 12) & pxh['mktcap'].notna()][['code', 'year', 'mktcap']]
tot = dec.groupby('year')['mktcap'].sum().rename('mktcap_tot')
rsize = dec.join(tot, on='year')
rsize['RSIZE'] = np.log(rsize['mktcap'] / rsize['mktcap_tot'])

mktvars = (fy[['code', 'year', 'EXRET', 'n_ret']]
           .merge(rsize[['code', 'year', 'RSIZE']], on=['code', 'year'], how='outer')
           .merge(hy, on=['code', 'year'], how='left'))
mktvars[['halt_frac_y', 'full_year']] = mktvars[['halt_frac_y', 'full_year']].fillna(0)
mktvars['full_year'] = mktvars['full_year'].astype(int)

fin = financials.copy()
fin['date'] = pd.to_datetime(fin['date'])
fin = fin.sort_values(['code', 'date']).reset_index(drop=True)
pq = pd.PeriodIndex(fin['date'], freq='Q')
fin['qtr'] = pq
fin['qidx'] = pq.year * 4 + (pq.quarter - 1)
g = fin.groupby('code', sort=False)
ok4 = (fin['qidx'] - g['qidx'].shift(3)) == 3
fin['ibq_ttm'] = np.where(
    ok4, g['ibq'].apply(lambda x: x.rolling(4, min_periods=4).sum())
    .reset_index(level=0, drop=True), np.nan)

acc = fin[fin['qtr'].dt.quarter == ACC_Q][['code', 'qtr', 'ibq_ttm', 'ltq', 'atq']].copy()
acc['year'] = acc['qtr'].dt.year
den = acc['atq'].where(acc['atq'] > 0)
acc['NITA'] = acc['ibq_ttm'] / den
acc['TLTA'] = acc['ltq'] / den
accvars = (acc[acc['year'].between(VY0, VY1)][['code', 'year', 'NITA', 'TLTA']]
           .sort_values(['code', 'year']).reset_index(drop=True))


def compute_fvol(df, deflator='saleq', fvol_cols=FVOL_COLS, diff_lag=4, win=8, min_obs=4):
    d = df.sort_values(['code', 'date']).reset_index(drop=True).copy()
    d['xoprq'] = d['cogsq'] + d['xsgaq']
    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)
    g = d.groupby('code', sort=False)
    has_lag = (d['qidx'] - g['qidx'].shift(diff_lag)) == diff_lag
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has_lag, d[c] - g[c].shift(diff_lag), np.nan)
    g = d.groupby('code', sort=False)
    for c in fvol_cols:
        d[f'std_{c}'] = (g[f'd_{c}'].apply(lambda s: s.rolling(win, min_periods=min_obs).std())
                         .reset_index(level=0, drop=True))
    d['defl_pos'] = d[deflator].where(d[deflator] > 0)
    g = d.groupby('code', sort=False)
    d['avg_defl'] = (g['defl_pos'].apply(lambda s: s.rolling(win, min_periods=min_obs).mean())
                     .reset_index(level=0, drop=True))
    d['avg_defl'] = d['avg_defl'].where(d['avg_defl'] > 0)
    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_defl']
        ind.append(f'fvol_{c}')
    return d, ind


def finalize_fvol(d, ind, fvol_cols=FVOL_COLS, min_valid=10):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid'] = d[rcols].notna().sum(axis=1)
    d['FVOL'] = d[rcols].mean(axis=1)
    d.loc[d['n_valid'] < min_valid, 'FVOL'] = np.nan
    return d


fv, ind = compute_fvol(financials.assign(date=pd.to_datetime(financials['date'])))
fv = finalize_fvol(fv, ind)
fv['qtr'] = pd.PeriodIndex(fv['date'], freq='Q')
fvol = fv.loc[fv['qtr'].dt.quarter == ACC_Q, ['code', 'qtr', 'FVOL']].copy()
fvol['year'] = fvol['qtr'].dt.year
fvol = fvol[fvol['year'].between(VY0, VY1)][['code', 'year', 'FVOL']].reset_index(drop=True)

ov = pd.DataFrame({
    '유효': [int(sigma['SIGMA_m12'].notna().sum()), int(sigma['SIGMA_d12'].notna().sum()),
           int(mktvars['EXRET'].notna().sum()), int(mktvars['RSIZE'].notna().sum()),
           int(accvars['NITA'].notna().sum()), int(accvars['TLTA'].notna().sum()),
           int(fvol['FVOL'].notna().sum())],
    '중앙': [round(sigma['SIGMA_m12_a'].median(), 4), round(sigma['SIGMA_d12_a'].median(), 4),
           round(mktvars['EXRET'].median(), 4), round(mktvars['RSIZE'].median(), 3),
           round(accvars['NITA'].median(), 4), round(accvars['TLTA'].median(), 4),
           round(fvol['FVOL'].median(), 4)]},
    index=pd.Index(['SIGMA_m12', 'SIGMA_d12', 'EXRET', 'RSIZE', 'NITA', 'TLTA', 'FVOL'],
                   name='변수'))
display(ov)

chk = pd.DataFrame({'값': [
    f'{VY0}~{VY1}',
    round(sigma[['SIGMA_m12', 'SIGMA_d12']].corr().iloc[0, 1], 3),
    int((accvars['TLTA'] > 1).sum()),
    int(mktvars.loc[mktvars['full_year'] == 1].shape[0]),
    int(sigma.loc[sigma['year'] == VY1, 'SIGMA_d12'].notna().sum()),
    int(fvol.loc[fvol['year'] == VY1, 'FVOL'].notna().sum())]},
    index=pd.Index(['변수 연도 범위', 'm12~d12 상관', 'TLTA > 1 (완전자본잠식)',
                    '연 완전정지 종목-연도',
                    f'{VY1}년 SIGMA_d12 유효', f'{VY1}년 FVOL 유효'], name='항목'))
display(chk)

,유효,중앙
변수,,
SIGMA_m12,17125,0.3404
SIGMA_d12,16729,0.4045
EXRET,16745,-0.0568
RSIZE,17143,-8.8250
NITA,16965,0.0272
TLTA,17386,0.4750
FVOL,16327,0.4826


,값
항목,
변수 연도 범위,2001~2025
m12~d12 상관,0.62
TLTA > 1 (완전자본잠식),162
연 완전정지 종목-연도,46
2025년 SIGMA_d12 유효,751
2025년 FVOL 유효,757


In [4]:
# 3

FILL_LIMIT = 2
BASE = ['NITA', 'TLTA', 'RSIZE', 'EXRET']
BF = [c + '_f' for c in BASE]
SF = 'SIGMA_d12_fill_a_f'
M5 = BF + [SF]
M6 = M5 + ['FVOL']

X = (sigma[['code', 'year', 'SIGMA_m12_a', 'SIGMA_d12_a']]
     .merge(mktvars[['code', 'year', 'EXRET', 'RSIZE', 'halt_frac_y', 'full_year']],
            on=['code', 'year'], how='outer')
     .merge(accvars[['code', 'year', 'NITA', 'TLTA']], on=['code', 'year'], how='outer'))

yrs = pd.DataFrame({'year': range(int(X['year'].min()), int(X['year'].max()) + 1)})
full = (X[['code']].drop_duplicates().merge(yrs, how='cross')
        .merge(X, on=['code', 'year'], how='left')
        .sort_values(['code', 'year']).reset_index(drop=True))

g = full.groupby('code', sort=False)
for c in BASE + ['SIGMA_m12_a', 'SIGMA_d12_a']:
    full[c + '_f'] = g[c].ffill(limit=FILL_LIMIT)

full['src'] = np.select(
    [full['SIGMA_d12_a'].notna(), full['SIGMA_d12_a_f'].notna()],
    ['실측', '이월'], default='결측')

need = full['SIGMA_d12_a_f'].isna() & full['SIGMA_m12_a_f'].notna()
full[SF] = full['SIGMA_d12_a_f'].where(~need, full['SIGMA_m12_a_f'])
full.loc[need, 'src'] = '월별 보완'

sub = full.groupby('year')[SF].transform('mean')
hit = full[SF].isna() & full['halt_frac_y'].fillna(0).gt(0) & sub.notna()
full.loc[hit, SF] = sub[hit]
full.loc[hit, 'src'] = '횡단면 평균'

full['obs_year'] = full['year'] + 1
panel = riskset.merge(full.drop(columns='year'), on=['code', 'obs_year'], how='left')

fv_a = fvol.copy()
fv_a['obs_year'] = fv_a['year'] + 1
panel = panel.merge(fv_a[['code', 'obs_year', 'FVOL']], on=['code', 'obs_year'], how='left')
panel['halt_frac_y'] = panel['halt_frac_y'].fillna(0.0)
panel['full_year'] = panel['full_year'].fillna(0).astype(int)
panel['src'] = panel['src'].fillna('결측')

KEEP = ['code', 'obs_year', 'y', 'censored', 'halt_frac_y', 'full_year', 'src'] + M6
panel = panel[KEEP].sort_values(['code', 'obs_year']).reset_index(drop=True)

p5 = panel[M5].notna().all(axis=1)
p6 = panel[M6].notna().all(axis=1)
past = panel['obs_year'] <= LAST_YEAR
tgt = panel['obs_year'] == TARGET

rows = []
for lab, m in [('위험집합', pd.Series(True, index=panel.index)),
               ('5변수 완비', p5), ('6변수 완비', p6)]:
    rows.append({'표본': lab,
                 '전체 행': int(m.sum()), '전체 사건': int(panel.loc[m, 'y'].sum()),
                 f'~{LAST_YEAR} 행': int((m & past).sum()),
                 f'~{LAST_YEAR} 사건': int(panel.loc[m & past, 'y'].sum()),
                 f'{TARGET} 행': int((m & tgt).sum())})
display(pd.DataFrame(rows).set_index('표본'))

rows = []
for k in ['실측', '이월', '월별 보완', '횡단면 평균', '결측']:
    m = (panel['src'] == k) & past
    if not m.sum():
        continue
    rows.append({'경로': k, '행': int(m.sum()), '사건': int(panel.loc[m, 'y'].sum()),
                 'SIGMA 중앙': round(panel.loc[m, SF].median(), 4)
                 if panel.loc[m, SF].notna().any() else np.nan,
                 '사건 SIGMA 중앙': round(panel.loc[m & panel['y'].eq(1), SF].median(), 4)
                 if (m & panel['y'].eq(1)).sum() else np.nan})
sp = pd.DataFrame(rows).set_index('경로')
display(sp)
print(f'전체 SIGMA 중앙 {panel[SF].median():.4f}')

byyr = (panel[p5].groupby('obs_year').agg(표본=('y', 'size'), 사건=('y', 'sum')))
byyr['6변수'] = panel[p6].groupby('obs_year').size()
byyr['사건률 %'] = (byyr['사건'] / byyr['표본'] * 100).round(2)
byyr.index.name = '관측연도'
display(byyr.tail(8))


,전체 행,전체 사건,~2025 행,~2025 사건,2026 행
표본,,,,,
위험집합,17050,113,16289,108,761
5변수 완비,16278,111,15528,106,750
6변수 완비,15497,91,14755,86,742


,행,사건,SIGMA 중앙,사건 SIGMA 중앙
경로,,,,
실측,15564,87,0.4050,0.9502
이월,37,8,0.8788,0.9273
월별 보완,302,12,0.4041,0.8290
횡단면 평균,4,1,0.4043,0.4043
결측,382,0,NaN,NaN


전체 SIGMA 중앙 0.4036


,표본,사건,6변수,사건률 %
관측연도,,,,
2019,688,0,679.0,0.00
2020,709,2,692.0,0.28
2021,709,0,709.0,0.00
2022,718,1,712.0,0.14
2023,735,1,725.0,0.14
2024,738,1,734.0,0.14
2025,745,2,733.0,0.27
2026,750,5,742.0,0.67


In [5]:
# 4

TRAIN0, TEST0 = 2002, 2008
WZ = 0.01
CUT = 0.05


def fit_predict(tr, te, cols):
    lo, hi = tr[cols].quantile(WZ), tr[cols].quantile(1 - WZ)
    Xtr = sm.add_constant(tr[cols].clip(lo, hi, axis=1).astype(float))
    Xte = sm.add_constant(te[cols].clip(lo, hi, axis=1).astype(float), has_constant='add')
    m = sm.Logit(tr['y'].astype(float), Xtr).fit(disp=0)
    return m, np.asarray(m.predict(Xte))


def walk(pan, cols, test0=TEST0, test1=None):
    test1 = LAST_YEAR if test1 is None else test1
    d = pan.dropna(subset=cols + ['y'])
    out = []
    for t in range(test0, test1 + 1):
        tr = d[(d['obs_year'] >= TRAIN0) & (d['obs_year'] < t)]
        te = d[d['obs_year'] == t]
        if tr['y'].sum() < 5 or len(te) == 0:
            continue
        o = te[['code', 'obs_year', 'y']].copy()
        _, o['ph'] = fit_predict(tr, te, cols)
        out.append(o)
    r = pd.concat(out, ignore_index=True)
    r['rk'] = r.groupby('obs_year')['ph'].rank(pct=True, ascending=False)
    r['dec'] = np.ceil(r['rk'] * 10).clip(1, 10).astype(int)
    return r


base = panel[panel[M6].notna().all(axis=1)]
oos5 = walk(base, M5)
oos6 = walk(base, M6)

rows = []
for lab, r in [('결합 5변수', oos5), ('6변수', oos6)]:
    ev = r[r['y'] == 1]
    hit = int((ev['rk'] <= CUT).sum())
    alert = r[r['rk'] <= CUT]
    rows.append({'모형': lab, '검증 관측': len(r), '검증 사건': int(r['y'].sum()),
                 'AUC': round(roc_auc_score(r['y'], r['ph']), 4),
                 '1분위 %': round((ev['dec'] == 1).mean() * 100, 1),
                 f'상위 {CUT:.0%} 포착': hit,
                 '재현율 %': round(hit / len(ev) * 100, 1),
                 '정밀도 %': round(hit / len(alert) * 100, 2),
                 '연평균 경보': round(len(alert) / r['obs_year'].nunique(), 1)})
display(pd.DataFrame(rows).set_index('모형'))

d6 = base.dropna(subset=M6 + ['y'])
tr = d6[(d6['obs_year'] >= TRAIN0) & (d6['obs_year'] < TARGET)]
te = d6[d6['obs_year'] == TARGET]
mod, ph = fit_predict(tr, te, M6)

watch = te[['code', 'obs_year']].copy()
watch['ph'] = ph
watch['rk'] = watch['ph'].rank(pct=True, ascending=False)
watch = watch.sort_values('ph', ascending=False).reset_index(drop=True)
watch['rank'] = watch.index + 1
watch['watch'] = (watch['rk'] <= CUT).astype(int)

print(f'{TARGET}년 명단  훈련 {len(tr)}관측 {int(tr["y"].sum())}사건  대상 {len(te)}종목  '
      f'상위 {CUT:.0%} = {int(watch["watch"].sum())}종목')
display(mod.params.round(3).rename('계수').to_frame())

w = watch[watch['watch'] == 1].copy()
w['확률 %'] = (w['ph'] * 100).round(2)
w['상위 %'] = (w['rk'] * 100).round(2)
display(w[['rank', 'code', '확률 %', '상위 %']].set_index('rank'))


,검증 관측,검증 사건,AUC,1분위 %,상위 5% 포착,재현율 %,정밀도 %,연평균 경보
모형,,,,,,,,
결합 5변수,11881,54,0.9712,88.9,44,81.5,7.52,32.5
6변수,11881,54,0.9813,92.6,46,85.2,7.86,32.5


2026년 명단  훈련 14755관측 86사건  대상 742종목  상위 5% = 37종목


,계수
const,-17.968
NITA_f,-0.053
TLTA_f,2.639
RSIZE_f,-0.520
EXRET_f,-0.767
SIGMA_d12_fill_a_f,4.253
FVOL,5.023


,code,확률 %,상위 %
rank,,,
1,A002410,47.41,0.13
2,A001140,45.24,0.27
3,A074610,30.44,0.40
4,A001470,28.85,0.54
5,A019490,23.38,0.67
6,A012170,22.15,0.81
7,A008500,17.69,0.94
8,A009310,16.40,1.08
9,A071950,14.83,1.21


In [6]:
# 5

name_map = delisted.set_index('code')['name']
tgt_panel = panel[panel['obs_year'] == TARGET].copy()

miss = pd.DataFrame({
    '결측': [int(tgt_panel[c].isna().sum()) for c in M6],
}, index=pd.Index(M6, name='변수'))
miss['결측 %'] = (miss['결측'] / len(tgt_panel) * 100).round(2)
print(f'{TARGET}년 위험집합 {len(tgt_panel)}종목의 변수 결측')
display(miss)

drop6 = tgt_panel[~tgt_panel[M6].notna().all(axis=1)].copy()
drop6['결측 변수'] = drop6[M6].isna().apply(lambda r: ','.join(r.index[r.values]), axis=1)
print(f'6변수 완비 탈락 {len(drop6)}종목의 결측 조합')
display(drop6['결측 변수'].value_counts().rename_axis('결측 변수').to_frame('종목'))

d5 = panel.dropna(subset=M5 + ['y'])
tr5 = d5[(d5['obs_year'] >= TRAIN0) & (d5['obs_year'] < TARGET)]
te5 = d5[d5['obs_year'] == TARGET]
_, ph5 = fit_predict(tr5, te5, M5)
w5 = te5[['code']].copy()
w5['ph5'] = ph5
w5['rk5'] = w5['ph5'].rank(pct=True, ascending=False)

ev = tgt_panel[tgt_panel['y'] == 1].copy()
ev = (ev.merge(watch[['code', 'ph', 'rk', 'rank']], on='code', how='left')
      .merge(w5, on='code', how='left'))
ev['종목'] = ev['code'].map(name_map)
ev['6변수 결측'] = ev[M6].isna().apply(lambda r: ','.join(r.index[r.values]) or '-', axis=1)
ev['6변수 확률 %'] = (ev['ph'] * 100).round(2)
ev['6변수 상위 %'] = (ev['rk'] * 100).round(2)
ev['5변수 확률 %'] = (ev['ph5'] * 100).round(2)
ev['5변수 상위 %'] = (ev['rk5'] * 100).round(2)
print(f'{TARGET}년 확정 부실 폐지 {len(ev)}건')
display(ev[['code', '종목', 'src', 'halt_frac_y', '6변수 결측',
            '6변수 확률 %', '6변수 상위 %', '5변수 확률 %', '5변수 상위 %']]
        .set_index('code'))

cmp5 = pd.DataFrame({
    '대상 종목': [len(watch), len(w5)],
    f'상위 {CUT:.0%}': [int((watch['rk'] <= CUT).sum()), int((w5['rk5'] <= CUT).sum())],
    '사건 포착': [int((ev['rk'] <= CUT).sum()), int((ev['rk5'] <= CUT).sum())],
    '확정 사건': [len(ev), len(ev)]},
    index=pd.Index(['6변수', '5변수'], name='모형'))
display(cmp5)


2026년 위험집합 761종목의 변수 결측


,결측,결측 %
변수,,
NITA_f,7,0.92
TLTA_f,2,0.26
RSIZE_f,0,0.00
EXRET_f,9,1.18
SIGMA_d12_fill_a_f,1,0.13
FVOL,17,2.23


6변수 완비 탈락 19종목의 결측 조합


,종목
결측 변수,
FVOL,8
EXRET_f,2
"NITA_f,TLTA_f,EXRET_f,FVOL",2
"NITA_f,EXRET_f,FVOL",2
"NITA_f,FVOL",2
"EXRET_f,FVOL",2
"NITA_f,EXRET_f,SIGMA_d12_fill_a_f,FVOL",1


2026년 확정 부실 폐지 5건


,종목,src,halt_frac_y,6변수 결측,6변수 확률 %,6변수 상위 %,5변수 확률 %,5변수 상위 %
code,,,,,,,,
A001140,국보,이월,1.0,-,45.24,0.27,36.19,0.27
A003560,IHQ,이월,1.0,-,2.14,5.12,1.28,7.47
A008110,대동전자,이월,1.0,-,0.16,25.61,0.12,49.60
A010600,웰바이오텍,이월,1.0,-,3.32,3.77,9.11,2.27
A033180,KH 필룩스,월별 보완,1.0,-,0.92,7.95,0.37,19.33


,대상 종목,상위 5%,사건 포착,확정 사건
모형,,,,
6변수,742,37,2,5
5변수,750,37,2,5


In [7]:
# 6

dec = halt_month[halt_month['ym'].dt.month == 12].copy()
dec['obs_year'] = dec['ym'].dt.year + 1
dec['tradable'] = (dec['full_halt'] == 0).astype(int)
trad = dec[['code', 'obs_year', 'tradable', 'halt_frac', 'full_halt']].rename(
    columns={'halt_frac': 'dec_halt_frac'})


def restrict(r):
    o = r.merge(trad[['code', 'obs_year', 'tradable', 'dec_halt_frac']],
                on=['code', 'obs_year'], how='left')
    o['tradable'] = o['tradable'].fillna(0).astype(int)
    o = o[o['tradable'] == 1].copy()
    o['rk'] = o.groupby('obs_year')['ph'].rank(pct=True, ascending=False)
    o['dec'] = np.ceil(o['rk'] * 10).clip(1, 10).astype(int)
    return o


rows = []
for lab, r in [('결합 5변수', oos5), ('6변수', oos6)]:
    for scope, s in [('전체', r), ('12월 거래가능', restrict(r))]:
        ev = s[s['y'] == 1]
        alert = s[s['rk'] <= CUT]
        rows.append({'모형': lab, '표본': scope, '검증 관측': len(s),
                     '검증 사건': int(s['y'].sum()),
                     'AUC': round(roc_auc_score(s['y'], s['ph']), 4),
                     f'상위 {CUT:.0%} 포착': int((ev['rk'] <= CUT).sum()),
                     '재현율 %': round((ev['rk'] <= CUT).mean() * 100, 1),
                     '1분위 %': round((ev['dec'] == 1).mean() * 100, 1),
                     '정밀도 %': round(ev['rk'].le(CUT).sum() / len(alert) * 100, 2)})
display(pd.DataFrame(rows).set_index(['모형', '표본']))

o6 = restrict(oos6)
lost = oos6[oos6['y'] == 1].merge(o6[['code', 'obs_year']].assign(keep=1),
                                  on=['code', 'obs_year'], how='left')
byyr = pd.DataFrame({
    '전체 사건': oos6[oos6['y'] == 1].groupby('obs_year').size(),
    '거래가능 사건': o6[o6['y'] == 1].groupby('obs_year').size()}).fillna(0).astype(int)
byyr.loc['합계'] = byyr.sum()
byyr.index.name = '검증연도'
display(byyr)

wt = watch.merge(trad[['code', 'obs_year', 'tradable', 'dec_halt_frac']],
                 on=['code', 'obs_year'], how='left')
wt['tradable'] = wt['tradable'].fillna(0).astype(int)
live_w = wt[wt['tradable'] == 1].copy()
live_w['rk'] = live_w['ph'].rank(pct=True, ascending=False)
live_w = live_w.sort_values('ph', ascending=False).reset_index(drop=True)
live_w['rank'] = live_w.index + 1
live_w['watch'] = (live_w['rk'] <= CUT).astype(int)

halted_w = wt[wt['tradable'] == 0].copy().sort_values('ph', ascending=False)
halted_w['종목'] = halted_w['code'].map(name_map)

print(f'{TARGET}년  6변수 대상 {len(wt)}종목  →  12월 거래가능 {len(live_w)}  '
      f'정지 {len(halted_w)}  /  상위 {CUT:.0%} = {int(live_w["watch"].sum())}종목')

lw = live_w[live_w['watch'] == 1].copy()
lw['확률 %'] = (lw['ph'] * 100).round(2)
lw['상위 %'] = (lw['rk'] * 100).round(2)
lw['12월 정지율'] = lw['dec_halt_frac'].round(3)
display(lw[['rank', 'code', '확률 %', '상위 %', '12월 정지율']].set_index('rank'))

halted_w['확률 %'] = (halted_w['ph'] * 100).round(2)
print(f'{TARGET}년 이미 정지 상태 (별도 관리) {len(halted_w)}종목')
display(halted_w[['code', '종목', '확률 %']].reset_index(drop=True))


검증 관측  검증 사건     AUC  상위 5% 포착  재현율 %  1분위 %  정밀도 %
모형     표본                                                           
결합 5변수 전체        11881     54  0.9712        44   81.5   88.9   7.52
       12월 거래가능  11803     42  0.9745        33   78.6   88.1   5.67
6변수    전체        11881     54  0.9813        46   85.2   92.6   7.86
       12월 거래가능  11803     42  0.9821        35   83.3   90.5   6.01

,전체 사건,거래가능 사건
검증연도,,
2008,1,1
2009,13,11
2010,11,11
2011,5,5
2012,5,5
2013,2,2
2014,1,1
2015,3,1
2016,2,2


2026년  6변수 대상 742종목  →  12월 거래가능 722  정지 20  /  상위 5% = 36종목


,code,확률 %,상위 %,12월 정지율
rank,,,,
1,A012170,22.15,0.14,0.000
2,A008500,17.69,0.28,0.476
3,A009310,16.40,0.42,0.000
4,A071950,14.83,0.55,0.000
5,A051630,9.02,0.69,0.000
6,A002880,8.12,0.83,0.000
7,A093240,8.08,0.97,0.000
8,A465770,7.41,1.11,0.000
9,A011300,7.17,1.25,0.000


2026년 이미 정지 상태 (별도 관리) 20종목


,code,종목,확률 %
0,A002410,NaN,47.41
1,A001140,국보,45.24
2,A074610,NaN,30.44
3,A001470,NaN,28.85
4,A019490,NaN,23.38
5,A001570,NaN,14.03
6,A000300,NaN,9.61
7,A006380,NaN,6.03
8,A002210,NaN,5.68
9,A020760,NaN,4.17


In [8]:
# 7

dl_t = delisted[pd.to_datetime(delisted['del_date']).dt.year == TARGET].copy()
dl_t['del_date'] = pd.to_datetime(dl_t['del_date'])

a = watch[['code', 'ph', 'rk']].rename(columns={'rk': 'rk_all'})
a['dec_all'] = np.ceil(a['rk_all'] * 10).clip(1, 10).astype(int)
b = live_w[['code', 'rk']].rename(columns={'rk': 'rk_live'})
b['dec_live'] = np.ceil(b['rk_live'] * 10).clip(1, 10).astype(int)

t = (dl_t[['code', 'name', 'del_date', 'cat', 'is_bad', 'excl']]
     .merge(trad[['code', 'obs_year', 'tradable', 'dec_halt_frac']]
            .query(f'obs_year == {TARGET}').drop(columns='obs_year'), on='code', how='left')
     .merge(a, on='code', how='left')
     .merge(b, on='code', how='left')
     .sort_values('del_date').reset_index(drop=True))

t['구분'] = np.where(t['excl'] == 1, '미분류', np.where(t['is_bad'] == 1, '부실', '비부실'))
t['12월 거래'] = np.where(t['tradable'].fillna(0) == 1, 'O', 'X')
t['12월 정지율'] = t['dec_halt_frac'].round(3)
t['확률 %'] = (t['ph'] * 100).round(2)
t['전체 상위 %'] = (t['rk_all'] * 100).round(2)
t['거래가능 상위 %'] = (t['rk_live'] * 100).round(2)
t['폐지'] = t['del_date'].dt.strftime('%Y-%m-%d')

out = t[['code', 'name', '폐지', 'cat', '구분', '12월 거래', '12월 정지율', '확률 %',
         '전체 상위 %', 'dec_all', '거래가능 상위 %', 'dec_live']]
out = out.rename(columns={'name': '종목', 'cat': '사유',
                          'dec_all': '전체 분위', 'dec_live': '거래가능 분위'})
print(f'{TARGET}년 상장폐지 {len(out)}건  (6변수 대상 {len(watch)}종목 / 거래가능 {len(live_w)}종목 기준)')
display(out.set_index('code'))

miss = t[t['ph'].isna()]
if len(miss):
    mp = panel[(panel['obs_year'] == TARGET) & panel['code'].isin(miss['code'])]
    print('6변수 표본에 없는 종목의 결측 내역')
    if len(mp):
        display(mp.set_index('code')[M6 + ['src', 'halt_frac_y']])
    print('위험집합에 없는 종목:', sorted(set(miss['code']) - set(mp['code'])))


2026년 상장폐지 10건  (6변수 대상 742종목 / 거래가능 722종목 기준)


,종목,폐지,사유,구분,12월 거래,12월 정지율,확률 %,전체 상위 %,전체 분위,거래가능 상위 %,거래가능 분위
code,,,,,,,,,,,
A450140,코오롱모빌리티그룹,2026-01-07,합병,비부실,O,0.000,0.14,28.71,3.0,26.73,3.0
A003560,IHQ,2026-01-15,감사의견,부실,X,1.000,2.14,5.12,1.0,NaN,NaN
A033180,KH 필룩스,2026-01-15,감사의견,부실,X,0.000,0.92,7.95,1.0,NaN,NaN
A010600,웰바이오텍,2026-01-26,감사의견,부실,X,1.000,3.32,3.77,1.0,NaN,NaN
A042670,HD현대인프라코어,2026-01-26,미분류,미분류,O,0.000,NaN,NaN,NaN,NaN,NaN
A001140,국보,2026-01-27,감사의견,부실,X,1.000,45.24,0.27,1.0,NaN,NaN
A019440,세아특수강,2026-02-12,합병,비부실,O,0.000,0.01,87.60,9.0,87.26,9.0
A008110,대동전자,2026-03-30,감사의견,부실,X,1.000,0.16,25.61,3.0,NaN,NaN
A138490,코오롱ENP,2026-04-16,합병,비부실,O,0.000,0.01,85.44,9.0,85.04,9.0


6변수 표본에 없는 종목의 결측 내역
위험집합에 없는 종목: ['A042670']


In [9]:
# 8

EVENT_CATS = ['부도', '정리절차', '자본잠식', '감사의견', '영업정지', '미제출',
              '기타부실', '주가시총', '유동성']

dmap = delisted[delisted['excl'] == 0].copy()
dmap['del_year'] = pd.to_datetime(dmap['del_date']).dt.year
dmap['is_ev'] = dmap['cat'].isin(EVENT_CATS).astype(int)
dmap = dmap[['code', 'del_year', 'cat', 'is_ev']]

panel2 = panel.merge(dmap, on='code', how='left')
panel2['y'] = ((panel2['del_year'] == panel2['obs_year']) & (panel2['is_ev'] == 1)).astype(int)

base2 = panel2[panel2[M6].notna().all(axis=1)]
oos6b = walk(base2, M6)
o6b = restrict(oos6b)

rows = []
for lab, r in [('기존 정의', oos6), ('손실 기준', oos6b)]:
    for scope, s in [('전체', r), ('12월 거래가능', restrict(r))]:
        ev = s[s['y'] == 1]
        alert = s[s['rk'] <= CUT]
        rows.append({'사건 정의': lab, '표본': scope, '검증 관측': len(s),
                     '검증 사건': int(s['y'].sum()),
                     'AUC': round(roc_auc_score(s['y'], s['ph']), 4),
                     f'상위 {CUT:.0%} 포착': int((ev['rk'] <= CUT).sum()),
                     '재현율 %': round((ev['rk'] <= CUT).mean() * 100, 1),
                     '1분위 %': round((ev['dec'] == 1).mean() * 100, 1),
                     '정밀도 %': round((ev['rk'] <= CUT).sum() / len(alert) * 100, 2)})
display(pd.DataFrame(rows).set_index(['사건 정의', '표본']))

yr = pd.DataFrame({
    '기존 전체': oos6[oos6['y'] == 1].groupby('obs_year').size(),
    '기존 거래가능': restrict(oos6).query('y == 1').groupby('obs_year').size(),
    '손실 전체': oos6b[oos6b['y'] == 1].groupby('obs_year').size(),
    '손실 거래가능': o6b[o6b['y'] == 1].groupby('obs_year').size()}).fillna(0).astype(int)
yr.loc['합계'] = yr.sum()
yr.index.name = '검증연도'
display(yr)

add = oos6b[(oos6b['y'] == 1)].merge(dmap[['code', 'cat']], on='code', how='left')
add = add[add['cat'].isin(['주가시총', '유동성'])]
addt = add.merge(trad[['code', 'obs_year', 'tradable']], on=['code', 'obs_year'], how='left')
addt['tradable'] = addt['tradable'].fillna(0).astype(int)
addt['상위 %'] = (addt['rk'] * 100).round(2)
print(f'형식요건으로 추가된 사건 {len(addt)}건 (12월 거래가능 {int(addt["tradable"].sum())}건)')
display(addt[['code', 'obs_year', 'cat', '상위 %', 'dec', 'tradable']]
        .sort_values('obs_year').reset_index(drop=True))


검증 관측  검증 사건     AUC  상위 5% 포착  재현율 %  1분위 %  정밀도 %
사건 정의 표본                                                           
기존 정의 전체        11881     54  0.9813        46   85.2   92.6   7.86
      12월 거래가능  11803     42  0.9821        35   83.3   90.5   6.01
손실 기준 전체        11881     55  0.9823        47   85.5   94.5   8.03
      12월 거래가능  11803     43  0.9836        36   83.7   93.0   6.19

,기존 전체,기존 거래가능,손실 전체,손실 거래가능
검증연도,,,,
2008,1,1,1,1
2009,13,11,13,11
2010,11,11,11,11
2011,5,5,5,5
2012,5,5,6,6
2013,2,2,2,2
2014,1,1,1,1
2015,3,1,3,1
2016,2,2,2,2


형식요건으로 추가된 사건 1건 (12월 거래가능 1건)


,code,obs_year,cat,상위 %,dec,tradable
0,A015110,2012,주가시총,0.98,1,1


In [10]:
# 9

dm0 = delisted.copy()
dm0['del_ym'] = pd.PeriodIndex(pd.to_datetime(dm0['del_date']), freq='M')

live_m = (px.loc[px[['ret', 'mktcap', 'price']].notna().any(axis=1), ['code', 'ym']]
          .drop_duplicates().assign(listed=1))
hs = halt_month.merge(live_m, on=['code', 'ym'], how='left')
hs['listed'] = hs['listed'].fillna((hs['n_days'] > 0).astype(int)).astype(int)
hs = hs.merge(dm0[['code', 'del_ym']], on='code', how='left')
hs = hs[(hs['listed'] == 1) & (hs['del_ym'].isna() | (hs['ym'] <= hs['del_ym']))]
hs = hs.sort_values(['code', 'ym']).reset_index(drop=True)
hs['fh'] = hs['full_halt'].astype(int)
hs['blk'] = ((hs['fh'] != hs.groupby('code')['fh'].shift())
             | (hs['code'] != hs['code'].shift())).cumsum()

epi = (hs[hs['fh'] == 1].groupby('blk')
       .agg(code=('code', 'first'), start=('ym', 'min'), end=('ym', 'max'),
            n_month=('ym', 'size')).reset_index(drop=True))
epi['year'] = epi['start'].dt.year

REOPEN_GAP = 3
ok = hs.loc[hs['fh'] == 0, ['code', 'ym', 'del_ym']].copy()
ok['ordm'] = ok['ym'].dt.year * 12 + ok['ym'].dt.month
lim = np.where(ok['del_ym'].notna(),
               ok['del_ym'].dt.year * 12 + ok['del_ym'].dt.month - REOPEN_GAP, np.inf)
ok = ok[ok['ordm'] <= lim]
last_ok = ok.groupby('code')['ym'].max().rename('last_trade')
epi = epi.merge(last_ok, on='code', how='left')
epi['재개'] = (epi['last_trade'].notna() & (epi['last_trade'] > epi['end'])).astype(int)

dm = delisted.copy()
dm['del_ym'] = pd.PeriodIndex(pd.to_datetime(dm['del_date']), freq='M')
dm['is_ev'] = dm['cat'].isin(EVENT_CATS).astype(int)
epi = epi.merge(dm[['code', 'del_ym', 'cat', 'is_ev', 'excl']], on='code', how='left')

epi['결과'] = np.where(epi['재개'] == 1, '재개',
               np.where(epi['del_ym'].notna() & (epi['del_ym'] >= epi['start']),
                        np.where(epi['is_ev'] == 1, '폐지(손실)', '폐지(비손실)'),
                        '표본 끝까지 정지'))
epi['정지~폐지(월)'] = ((epi['del_ym'].dt.year * 12 + epi['del_ym'].dt.month)
                     - (epi['start'].dt.year * 12 + epi['start'].dt.month))

epi = epi[epi['year'].between(YR0, LAST_YEAR)]

print(f'완전정지 에피소드 {len(epi)}건  종목 {epi["code"].nunique()}')
r = epi.groupby('결과').agg(건수=('code', 'size'), 종목=('code', 'nunique'),
                          정지개월_중앙=('n_month', 'median'))
r['비율 %'] = (r['건수'] / len(epi) * 100).round(1)
r.index.name = '결과'
display(r)

yr = (epi.groupby('year')
      .agg(진입=('code', 'size'),
           재개=('재개', 'sum')))
yr['폐지(손실)'] = epi[epi['결과'] == '폐지(손실)'].groupby('year').size()
yr['정지 유지'] = epi[epi['결과'] == '표본 끝까지 정지'].groupby('year').size()
yr = yr.fillna(0).astype(int)
yr['재개 %'] = (yr['재개'] / yr['진입'] * 100).round(1)
yr.loc['합계'] = [yr['진입'].sum(), yr['재개'].sum(), yr['폐지(손실)'].sum(),
                yr['정지 유지'].sum(), round(yr['재개'].sum() / yr['진입'].sum() * 100, 1)]
yr.index.name = '진입 연도'
display(yr)

lsrc = epi[epi['결과'].str.startswith('폐지')]
lag = lsrc['정지~폐지(월)'].dropna()
print(f'폐지로 끝난 에피소드 {len(lsrc)}건, 개월 산출 {len(lag)}건')
if len(lag):
    display(pd.DataFrame({'값': [len(lag), lag.median(), lag.quantile(.25),
                                 lag.quantile(.75), lag.max()]},
                         index=pd.Index(['건수', '중앙', '25%', '75%', '최대'],
                                        name='첫 정지 → 폐지 (개월)')).round(1))

ent = epi[['code', 'year']].drop_duplicates().assign(y_halt=1)
ent['obs_year'] = ent['year']
lbl = panel[['code', 'obs_year', 'y']].merge(
    ent[['code', 'obs_year', 'y_halt']], on=['code', 'obs_year'], how='left')
lbl['y_halt'] = lbl['y_halt'].fillna(0).astype(int)
byyr = lbl[lbl['obs_year'].between(TEST0, LAST_YEAR)].groupby('obs_year').agg(
    위험집합=('y', 'size'), 폐지=('y', 'sum'), 정지진입=('y_halt', 'sum'))
byyr.loc['합계'] = byyr.sum()
byyr.index.name = '관측연도'
display(byyr)


완전정지 에피소드 320건  종목 240


,건수,종목,정지개월_중앙,비율 %
결과,,,,
재개,201,149,1.0,62.8
폐지(비손실),51,51,1.0,15.9
폐지(손실),55,55,5.0,17.2
표본 끝까지 정지,13,13,15.0,4.1


,진입,재개,폐지(손실),정지 유지,재개 %
진입 연도,,,,,
2002,15.0,10.0,3.0,0.0,66.7
2003,13.0,9.0,2.0,0.0,69.2
2004,14.0,8.0,5.0,0.0,57.1
2005,17.0,9.0,6.0,0.0,52.9
2006,6.0,2.0,2.0,0.0,33.3
2007,6.0,2.0,0.0,0.0,33.3
2008,10.0,7.0,2.0,0.0,70.0
2009,11.0,9.0,1.0,0.0,81.8
2010,9.0,4.0,4.0,0.0,44.4


폐지로 끝난 에피소드 106건, 개월 산출 106건


,값
첫 정지 → 폐지 (개월),
건수,106.0
중앙,1.0
25%,0.0
75%,9.5
최대,49.0


,위험집합,폐지,정지진입
관측연도,,,
2008,632,1,9
2009,648,13,10
2010,656,11,9
2011,661,5,17
2012,662,5,11
2013,663,3,18
2014,667,1,16
2015,680,3,10
2016,691,2,14


In [11]:
# 10

HALT_NS = [3, 5, 8, 10, 15, 20, 30]

d0 = daily_returns.copy()
d0['date'] = pd.to_datetime(d0['date'])
d0 = d0.sort_values(['code', 'date']).reset_index(drop=True)
d0['ym'] = pd.PeriodIndex(d0['date'], freq='M')
d0['year'] = d0['date'].dt.year
d0['is0'] = (d0['dret'] == 0).astype(int)
d0['grp'] = (d0['is0'] != d0.groupby('code')['is0'].shift()).cumsum()
d0['ylen'] = d0.groupby(['grp', 'year'])['is0'].transform('size')

pxf = px.sort_values(['code', 'date']).copy()
pxf['p_prev'] = pxf.groupby('code')['price'].shift(1)
pxf['frozen'] = (pxf['price'].notna() & pxf['p_prev'].notna()
                 & ((pxf['price'] - pxf['p_prev']).abs() < 1e-9)).astype(int)
pxf['no_daily'] = (~pxf['code'].isin(set(d0['code']))).astype(int)
pxb = pxf[['code', 'ym', 'ret', 'frozen', 'no_daily']]

dm1 = delisted.copy()
dm1['del_ym'] = pd.PeriodIndex(pd.to_datetime(dm1['del_date']), freq='M')
dm1['is_ev'] = dm1['cat'].isin(EVENT_CATS).astype(int)
live_m1 = (px.loc[px[['ret', 'mktcap', 'price']].notna().any(axis=1), ['code', 'ym']]
           .drop_duplicates().assign(listed=1))


def build_epi(n, gap=3):
    h = (pd.DataFrame({'code': d0['code'], 'ym': d0['ym'],
                       'h': ((d0['is0'] == 1) & (d0['ylen'] >= n)).astype(int)})
         .groupby(['code', 'ym']).agg(n_days=('h', 'size'), n_halt=('h', 'sum'))
         .reset_index())
    h['halt_frac'] = h['n_halt'] / h['n_days']
    h = pxb.merge(h, on=['code', 'ym'], how='outer')
    for c, v in [('halt_frac', 0.0), ('n_days', 0), ('frozen', 0)]:
        h[c] = h[c].fillna(v)
    h['no_daily'] = h['no_daily'].fillna(1).astype(int)
    by_d = h['halt_frac'] >= FULL_HALT
    by_p = (h['no_daily'] == 1) & (h['frozen'] == 1)
    reopen = (by_d | by_p) & h['ret'].notna() & h['ret'].abs().gt(1e-9)
    h['fh'] = ((by_d | by_p) & ~reopen).astype(int)

    h = h.merge(live_m1, on=['code', 'ym'], how='left')
    h['listed'] = h['listed'].fillna((h['n_days'] > 0).astype(int)).astype(int)
    h = h.merge(dm1[['code', 'del_ym']], on='code', how='left')
    h = h[(h['listed'] == 1) & (h['del_ym'].isna() | (h['ym'] <= h['del_ym']))]
    h = h.sort_values(['code', 'ym']).reset_index(drop=True)
    h['blk'] = ((h['fh'] != h.groupby('code')['fh'].shift())
                | (h['code'] != h['code'].shift())).cumsum()

    e = (h[h['fh'] == 1].groupby('blk')
         .agg(code=('code', 'first'), start=('ym', 'min'), end=('ym', 'max'),
              n_month=('ym', 'size')).reset_index(drop=True))
    e['year'] = e['start'].dt.year

    o = h.loc[h['fh'] == 0, ['code', 'ym', 'del_ym']].copy()
    o['ordm'] = o['ym'].dt.year * 12 + o['ym'].dt.month
    lim = np.where(o['del_ym'].notna(),
                   o['del_ym'].dt.year * 12 + o['del_ym'].dt.month - gap, np.inf)
    o = o[o['ordm'] <= lim]
    e = e.merge(o.groupby('code')['ym'].max().rename('last_trade'), on='code', how='left')
    e['재개'] = (e['last_trade'].notna() & (e['last_trade'] > e['end'])).astype(int)
    e = e.merge(dm1[['code', 'del_ym', 'is_ev']], on='code', how='left')
    e['결과'] = np.where(e['재개'] == 1, '재개',
                np.where(e['del_ym'].notna() & (e['del_ym'] >= e['start']),
                         np.where(e['is_ev'] == 1, '폐지(손실)', '폐지(비손실)'),
                         '표본 끝까지 정지'))
    return e[e['year'].between(YR0, LAST_YEAR)], int(h['fh'].sum())


ORD = ['재개', '폐지(손실)', '폐지(비손실)', '표본 끝까지 정지']
rows, byyr = [], {}
for n in HALT_NS:
    e, nfh = build_epi(n)
    v = e['결과'].value_counts().reindex(ORD).fillna(0).astype(int)
    rows.append({'기준일수': n, '완전정지 월': nfh, '에피소드': len(e),
                 **{k: int(v[k]) for k in ORD},
                 '재개 %': round(v['재개'] / len(e) * 100, 1)})
    byyr[n] = e.groupby('year').size()
display(pd.DataFrame(rows).set_index('기준일수'))

t = pd.DataFrame(byyr).fillna(0).astype(int)
t.columns = [f'{n}일' for n in t.columns]
t.index.name = '진입 연도'
display(t.tail(10))


,완전정지 월,에피소드,재개,폐지(손실),폐지(비손실),표본 끝까지 정지,재개 %
기준일수,,,,,,,
3,1910,322,201,55,53,13,62.4
5,1910,322,201,55,53,13,62.4
8,1908,320,201,55,51,13,62.8
10,1906,318,201,55,49,13,63.2
15,1875,292,201,55,23,13,68.8
20,1860,280,201,55,11,13,71.8
30,1786,225,156,49,8,12,69.3


,3일,5일,8일,10일,15일,20일,30일
진입 연도,,,,,,,
2016,19,19,19,19,16,16,13
2017,14,14,14,14,13,12,7
2018,13,13,13,13,12,11,8
2019,8,8,8,8,8,8,8
2020,15,15,15,15,14,13,12
2021,11,11,11,11,10,10,10
2022,8,8,8,8,7,6,6
2023,12,12,12,12,12,12,10
2024,13,13,13,13,12,12,11


In [12]:
# 11

ent = (epi[['code', 'year']].drop_duplicates()
       .rename(columns={'year': 'obs_year'}).assign(y_halt=1))

ph = panel.merge(ent, on=['code', 'obs_year'], how='left')
ph['y_halt'] = ph['y_halt'].fillna(0).astype(int)
ph = ph.merge(trad[['code', 'obs_year', 'tradable']], on=['code', 'obs_year'], how='left')
ph['tradable'] = ph['tradable'].fillna(0).astype(int)
ph = ph[ph['tradable'] == 1].copy()
ph['y'] = ph['y_halt']

baseh = ph[ph[M6].notna().all(axis=1)]
oosh5 = walk(baseh, M5)
oosh6 = walk(baseh, M6)

rows = []
for lab, r in [('결합 5변수', oosh5), ('6변수', oosh6)]:
    ev = r[r['y'] == 1]
    alert = r[r['rk'] <= CUT]
    rows.append({'모형': lab, '검증 관측': len(r), '검증 사건': int(r['y'].sum()),
                 '기저 %': round(r['y'].mean() * 100, 2),
                 'AUC': round(roc_auc_score(r['y'], r['ph']), 4),
                 f'상위 {CUT:.0%} 포착': int((ev['rk'] <= CUT).sum()),
                 '재현율 %': round((ev['rk'] <= CUT).mean() * 100, 1),
                 '1분위 %': round((ev['dec'] == 1).mean() * 100, 1),
                 '정밀도 %': round((ev['rk'] <= CUT).sum() / len(alert) * 100, 2),
                 '리프트': round(((ev['rk'] <= CUT).sum() / len(alert)) / r['y'].mean(), 1)})
display(pd.DataFrame(rows).set_index('모형'))

yr = (oosh6.groupby('obs_year')
      .agg(대상=('y', 'size'), 정지진입=('y', 'sum')))
yr['상위 5% 포착'] = oosh6[(oosh6['y'] == 1) & (oosh6['rk'] <= CUT)].groupby('obs_year').size()
yr = yr.fillna(0).astype(int)
yr.loc['합계'] = yr.sum()
yr.index.name = '검증연도'
display(yr)

dh = baseh.dropna(subset=M6 + ['y'])
trh = dh[(dh['obs_year'] >= TRAIN0) & (dh['obs_year'] < TARGET)]
teh = dh[dh['obs_year'] == TARGET]
modh, phh = fit_predict(trh, teh, M6)

wh = teh[['code']].copy()
wh['ph'] = phh
wh['rk'] = wh['ph'].rank(pct=True, ascending=False)
wh = wh.sort_values('ph', ascending=False).reset_index(drop=True)
wh['rank'] = wh.index + 1
print(f'{TARGET}년 정지 진입 명단  훈련 {len(trh)}관측 {int(trh["y"].sum())}사건  '
      f'대상 {len(teh)}종목  상위 {CUT:.0%} = {int((wh["rk"] <= CUT).sum())}종목')
display(modh.params.round(3).rename('계수').to_frame())

top = wh[wh['rk'] <= CUT].copy()
top['확률 %'] = (top['ph'] * 100).round(2)
top['상위 %'] = (top['rk'] * 100).round(2)
display(top[['rank', 'code', '확률 %', '상위 %']].set_index('rank').head(20))

,검증 관측,검증 사건,기저 %,AUC,상위 5% 포착,재현율 %,1분위 %,정밀도 %,리프트
모형,,,,,,,,,
결합 5변수,11803,208,1.76,0.7257,82,39.4,46.2,14.09,8.0
6변수,11803,208,1.76,0.7323,86,41.3,49.0,14.78,8.4


,대상,정지진입,상위 5% 포착
검증연도,,,
2008,585,9,3
2009,597,9,3
2010,604,9,2
2011,602,16,6
2012,612,11,7
2013,628,18,6
2014,636,16,8
2015,639,10,6
2016,647,14,9


2026년 정지 진입 명단  훈련 14661관측 257사건  대상 722종목  상위 5% = 36종목


,계수
const,-7.063
NITA_f,-2.118
TLTA_f,2.093
RSIZE_f,-0.023
EXRET_f,-0.451
SIGMA_d12_fill_a_f,1.785
FVOL,1.338


,code,확률 %,상위 %
rank,,,
1,A071950,18.10,0.14
2,A012170,16.87,0.28
3,A069640,13.01,0.42
4,A047400,12.76,0.55
5,A013360,12.36,0.69
6,A009310,11.62,0.83
7,A017040,11.33,0.97
8,A008600,10.22,1.11
9,A066970,10.09,1.25
